# NB_hospital_daily_summary

Builds `adb_caresync_project.gold.hospital_daily_summary` (one row per hospital per calendar day) from silver lab_results, hospitals and calendar, per STTM Silver to Gold v1.0. Includes day-over-day test volume trend via LAG window function. Load type: full rebuild (CREATE OR REPLACE TABLE AS SELECT).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync_project.gold;

CREATE OR REPLACE TABLE adb_caresync_project.gold.hospital_daily_summary AS
WITH daily_agg AS (
  -- Aggregate lab results to hospital + day grain
  SELECT
    lr.hospital_id,
    lr.test_date                                              AS summary_date,
    COUNT(*)                                                  AS total_tests,
    COUNT(DISTINCT lr.patient_id)                              AS unique_patients_tested,
    COUNT(DISTINCT lr.doctor_id)                               AS active_doctor_count,
    COUNT_IF(lr.result_status = 'Abnormal')                    AS abnormal_test_count,
    COUNT_IF(lr.result_status = 'Critical')                    AS critical_test_count
  FROM adb_caresync_project.silver.lab_results lr
  GROUP BY lr.hospital_id, lr.test_date
),
with_window AS (
  -- Add LAG window function for day-over-day comparison
  SELECT
    da.*,
    LAG(da.total_tests) OVER (PARTITION BY da.hospital_id ORDER BY da.summary_date) AS prev_day_total_tests
  FROM daily_agg da
)
SELECT
  CAST(w.hospital_id AS INT)                                                      AS hospital_id,
  CAST(h.hospital_name AS STRING)                                                 AS hospital_name,
  CAST(w.summary_date AS DATE)                                                    AS summary_date,
  CAST(c.day_name AS STRING)                                                      AS day_name,
  CAST(c.is_weekend AS BOOLEAN)                                                   AS is_weekend,
  CAST(w.total_tests AS INT)                                                      AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                           AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                              AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                              AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                              AS critical_test_count,
  CAST(ROUND(w.abnormal_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2))  AS abnormal_rate_pct,
  CAST(ROUND(w.critical_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2))  AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                             AS prev_day_total_tests,
  CAST(
    ROUND(
      CASE
        WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
        ELSE (w.total_tests - w.prev_day_total_tests) * 100.0 / w.prev_day_total_tests
      END, 2
    ) AS DECIMAL(6,2)
  )                                                                               AS test_volume_change_pct
FROM with_window w
LEFT JOIN adb_caresync_project.silver.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN adb_caresync_project.silver.calendar c
  ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT);

In [0]:
updated_df = spark.table("adb_caresync_project.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))